# Correction 03 : comprendre KNN avec Iris

On utilise Iris pour rendre visible l'idée de KNN : comparer une nouvelle fleur aux fleurs connues, puis voter parmi les voisines les plus proches.

## 1. Charger le dataset Iris

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X.head()

In [ ]:
iris.target_names

`X` contient les `4` mesures des fleurs.

`y` contient l'espèce sous forme numérique :

- `0` : setosa ;
- `1` : versicolor ;
- `2` : virginica.

## 2. Séparer train et test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

Le train contient `120` fleurs.

Le test contient `30` fleurs.

Le modèle apprend avec le train et on vérifie avec le test.

## 3. Entraîner un modèle KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

modele = KNeighborsClassifier(n_neighbors=5)
modele.fit(X_train, y_train)

`n_neighbors=5` signifie que KNN regarde les `5` fleurs les plus proches pour voter.

Cela ne signifie pas qu'il utilise 5 critères. Ici, il utilise bien les `4` mesures disponibles.

## 4. Tester rapidement le modèle

In [ ]:
score_test = modele.score(X_test, y_test)

print("Score test :", score_test)

Le score test mesure la proportion de fleurs bien classées sur des données non vues pendant l'entraînement.

## 5. Prendre une fleur inconnue pour le modèle

In [ ]:
index = X_test.index[0]
nouvelle_fleur = X_test.loc[[index]]
vraie_classe = y_test.loc[index]

nouvelle_fleur

In [ ]:
prediction = modele.predict(nouvelle_fleur)

print("Vraie espèce   :", iris.target_names[vraie_classe])
print("Espèce prédite :", iris.target_names[prediction[0]])

## 6. Observer les voisins utilisés par KNN

In [ ]:
distances, indices = modele.kneighbors(nouvelle_fleur)

voisins = X_train.iloc[indices[0]].copy()
voisins["distance"] = distances[0]
voisins["classe"] = y_train.iloc[indices[0]].values
voisins["espece"] = [iris.target_names[c] for c in voisins["classe"]]

voisins

KNN a comparé la nouvelle fleur à toutes les fleurs du train, puis il a gardé les `5` distances les plus petites.

## 7. Comprendre le vote

In [ ]:
vote = voisins["espece"].value_counts()

vote

La classe prédite est celle qui apparaît le plus souvent parmi les voisins.

C'est le coeur de KNN en classification :

```text
proximité → voisins → vote → prédiction
```